# EBSD To Texture Outputs

This notebook demonstrates the new practical path from vectorized orientation
construction through a normalized EBSD-like dataset into ODF, PF, and IPF outputs.

The example is intentionally self-contained: it uses an in-memory phase definition
and a stub KikuchiPy-like signal so the workflow can be read as pure PyTex surface
area rather than as package-integration ceremony.


In [ ]:
from pathlib import Path
import tempfile

import numpy as np

from pytex import (
    AcquisitionGeometry,
    AtomicSite,
    BenchmarkManifest,
    build_crystal_scene,
    CalibrationRecord,
    CrystalCellOverlay,
    CrystalDirection,
    CrystalDirectionOverlay,
    CrystalMap,
    CrystalPlane,
    CrystalPlaneOverlay,
    DirectionAnnotationStyle,
    DiffractionGeometry,
    EulerSet,
    ExperimentManifest,
    FrameDomain,
    FrameTransform,
    Handedness,
    InversePoleFigure,
    KernelSpec,
    KinematicSimulation,
    Lattice,
    get_phase_fixture,
    list_phase_fixtures,
    list_style_themes,
    MeasurementQuality,
    MillerIndex,
    ODF,
    Orientation,
    OrientationRelationship,
    OrientationSet,
    Phase,
    PhaseTransformationRecord,
    PoleFigure,
    PowderPattern,
    PowderReflection,
    ReferenceFrame,
    read_validation_manifest,
    read_workflow_result_manifest,
    resolve_style,
    RadiationSpec,
    Rotation,
    ScatteringSetup,
    SymmetrySpec,
    TransformationVariant,
    UnitCell,
    ValidationManifest,
    VectorSet,
    WorkflowResultManifest,
    ZoneAxis,
    PlaneAnnotationStyle,
    generate_saed_pattern,
    generate_xrd_pattern,
    normalize_ebsd,
    plot_odf,
    plot_crystal_structure_3d,
    plot_inverse_pole_figure,
    plot_ipf_map,
    plot_orientations,
    plot_kam_map,
    plot_pole_figure,
    plot_saed_pattern,
    plot_symmetry_elements,
    plot_symmetry_orbit,
    plot_vector_set,
    plot_xrd_pattern,
)


def make_crystal_frame():
    return ReferenceFrame(
        "crystal",
        FrameDomain.CRYSTAL,
        ("a", "b", "c"),
        Handedness.RIGHT,
    )


def make_context():
    crystal = make_crystal_frame()
    specimen = ReferenceFrame(
        "specimen",
        FrameDomain.SPECIMEN,
        ("x", "y", "z"),
        Handedness.RIGHT,
    )
    map_frame = ReferenceFrame(
        "map",
        FrameDomain.MAP,
        ("i", "j", "k"),
        Handedness.RIGHT,
    )
    detector = ReferenceFrame(
        "detector",
        FrameDomain.DETECTOR,
        ("u", "v", "n"),
        Handedness.RIGHT,
    )
    lab = ReferenceFrame(
        "lab",
        FrameDomain.LABORATORY,
        ("X", "Y", "Z"),
        Handedness.RIGHT,
    )
    phase = get_phase_fixture("ni_fcc").load_phase(crystal_frame=crystal)
    return crystal, specimen, map_frame, detector, lab, phase


def describe_phase_fixture(fixture_id):
    record = get_phase_fixture(fixture_id)
    return {
        "fixture_id": record.fixture_id,
        "display_name": record.display_name,
        "artifact_path": str(record.artifact_path),
        "metadata_path": str(record.metadata_path),
        "intended_uses": tuple(record.metadata["intended_uses"]),
    }


def load_zr_hcp_phase():
    return get_phase_fixture("zr_hcp").load_phase(crystal_frame=make_crystal_frame())


def load_diamond_phase():
    return get_phase_fixture("diamond").load_phase(crystal_frame=make_crystal_frame())


def publication_crystal_style():
    return {
        "crystal": {
            "atom_radius_scale": 0.5,
            "atom_edgewidth": 0.0,
            "atom_surface_resolution": 34,
            "bond_surface_resolution": 28,
            "bond_alpha": 0.72,
            "bond_color": "#7c8ea3",
            "atom_specular_strength": 0.42,
            "light_specular": 0.4,
        }
    }


In [ ]:
crystal = make_crystal_frame()
specimen = ReferenceFrame(
    "specimen",
    FrameDomain.SPECIMEN,
    ("x", "y", "z"),
    Handedness.RIGHT,
)
symmetry = SymmetrySpec.from_point_group("m-3m", reference_frame=crystal)
lattice = Lattice(3.6, 3.6, 3.6, 90.0, 90.0, 90.0, crystal_frame=crystal)
phase = Phase("fcc_demo", lattice=lattice, symmetry=symmetry, crystal_frame=crystal)

orientations = OrientationSet.from_plane_direction(
    plane=np.array(
        [
            [0, 0, 1],
            [0, 0, 1],
            [1, 1, 1],
            [1, 1, 1],
        ]
    ),
    direction=np.array(
        [
            [1, 0, 0],
            [0, 1, 0],
            [1, -1, 0],
            [1, 0, -1],
        ]
    ),
    specimen_frame=specimen,
    phase=phase,
)

axes, angles = orientations.to_axes_angles()
rodrigues = orientations.as_rotation_set().to_rodrigues(frank=True)

print("orientation count:", len(orientations))
print("first axis-angle:", axes[0], angles[0])
print("first Rodrigues-Frank:", rodrigues[0])


In [ ]:
class XMapStub:
    def __init__(self, quaternions):
        self.rotations = quaternions
        self.x = np.array([0.0, 1.0, 0.0, 1.0], dtype=np.float64)
        self.y = np.array([0.0, 0.0, 1.0, 1.0], dtype=np.float64)
        self.shape = (2, 2)
        self.dx = 1.0
        self.dy = 1.0
        self.source_file = "stub-indexed.h5"


class SignalStub:
    def __init__(self, xmap):
        self.xmap = xmap


dataset = normalize_ebsd(
    SignalStub(XMapStub(orientations.quaternions)),
    frames=(crystal, specimen, specimen),
    phase=phase,
)

print(dataset.manifest.phase_name)
print(dataset.crystal_map.grid_shape)


In [ ]:
report = dataset.crystal_map.texture_report(
    poles=([1, 0, 0], [1, 1, 1]),
    sample_directions=("x", "z"),
    plot=False,
)

print("pole figures:", len(report.pole_figures))
print("inverse pole figures:", len(report.inverse_pole_figures))
print("ODF weights:", report.odf.normalized_weights)


In [ ]:
odf_figure = plot_odf(report.odf, kind="sections", section_phi2_deg=(0.0, 45.0))
pole_figure = plot_pole_figure(report.pole_figures[0], kind="contour")
inverse_pole_figure = plot_inverse_pole_figure(report.inverse_pole_figures[0])
ipf_map = plot_ipf_map(dataset.crystal_map, direction="z")
kam_map = plot_kam_map(dataset.crystal_map)

odf_figure


In [ ]:
pole_figure


In [ ]:
inverse_pole_figure


In [ ]:
ipf_map


In [ ]:
kam_map


The important part is the continuity of semantic objects:

- plane/direction pairs construct orientations without dropping phase or symmetry
- `normalize_ebsd(...)` returns a `NormalizedEBSDDataset`
- `CrystalMap` produces ODF, PF, and IPF outputs directly
- the same `CrystalMap` also feeds IPF-map and KAM-map plotting

That keeps the whole EBSD-to-texture path inside one explicit PyTex data model.
